# ReAct Prompting — Ollama (gemma4:31b)
Local inference via Ollama. No API key required. GPU-accelerated on your RTX A6000.


In [1]:
import requests, json, base64, os, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
print('Imports OK')
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama running. Models: {models}')
except Exception as e:
    print(f'ERROR: Ollama not reachable — run: ollama serve ({e})')


Imports OK
Ollama running. Models: ['gemma4:31b']


# ReAct Prompting
Follows a structured Thought → Action → Observation → Decision cycle.
- **Thought 1**: Pre-analysis reasoning (what to look for)
- **Action 1**: Frame examination (with images, batched)
- **Observation 1**: Consolidated findings
- **Thought 2**: Pattern analysis
- **Action 2**: Focused re-analysis
- **Thought 3**: Crime classification reasoning
- **Final Answer**: Structured report


In [ ]:
import os, json, base64, requests, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================================================================
#  CONFIGURATION
# ================================================================
OLLAMA_URL     = 'http://localhost:11434/api/chat'
MODEL_NAME     = 'gemma4:31b'
FRAMES_DIR     = r'C:\Opeyemi\PROMPTS\FRAMES'
RESULTS_BASE   = r'C:\Opeyemi\PROMPTS\RESULTS'
FRAME_EXT      = '.jpg'
FRAME_INTERVAL = 1
BATCH_SIZE     = 20
MAX_WORKERS    = 4
SAVE_DIR       = r'C:\Opeyemi\PROMPTS\RESULTS\OLLAMA\REACT'
os.makedirs(SAVE_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(SAVE_DIR, 'react_ollama_checkpoint.json')

# ================================================================
#  FRAME HELPERS
# ================================================================
def extract_frame_number(filename):
    import re as _re
    name = os.path.splitext(filename)[0]
    m = _re.search(r'frame[_\-]?(\d+)', name, _re.IGNORECASE)
    if m: return int(m.group(1))
    nums = _re.findall(r'\d+', name)
    return int(nums[-1]) if nums else 0

def discover_all_videos_and_frames(frames_dir=None):
    if frames_dir is None: frames_dir = FRAMES_DIR
    print(f'\n=== DISCOVERING FRAMES ===')
    print(f'    Root : {frames_dir}')
    all_videos = {}
    if not os.path.isdir(frames_dir):
        print(f'  ERROR: FRAMES_DIR not found: {frames_dir}'); return all_videos
    crime_types = sorted([d for d in os.listdir(frames_dir)
        if os.path.isdir(os.path.join(frames_dir, d)) and not d.startswith('_')])
    print(f'  Categories : {crime_types}')
    for crime_type in crime_types:
        cat_dir = os.path.join(frames_dir, crime_type)
        video_stems = sorted([d for d in os.listdir(cat_dir)
            if os.path.isdir(os.path.join(cat_dir, d))])
        print(f'    {crime_type:20s}: {len(video_stems)} videos')
        for video_stem in video_stems:
            vdir = os.path.join(cat_dir, video_stem)
            frame_files = sorted(
                [ff for ff in os.listdir(vdir) if ff.lower().endswith(FRAME_EXT)],
                key=extract_frame_number)
            if not frame_files:
                print(f'      WARNING: no frames in {vdir} - skipping'); continue
            key = f'{crime_type}_{video_stem}'
            all_videos[key] = {'crime_type': crime_type, 'video_id': video_stem,
                               'frames_dir': vdir, 'frames': frame_files}
    print(f'  Total videos ready: {len(all_videos)}')
    return all_videos

def load_frames_for_video(video_info, frame_interval=1):
    frames_data = {}
    vdir = video_info['frames_dir']
    frame_files = video_info['frames']
    video_id = video_info['video_id']
    selected = frame_files[::frame_interval]
    label = 'ALL' if frame_interval == 1 else f'every {frame_interval}th'
    print(f'  Loading {len(selected)} frames ({label}) for {video_id} ...')
    for ff in selected:
        fp = os.path.join(vdir, ff)
        try:
            with open(fp, 'rb') as fh:
                frames_data[ff] = base64.b64encode(fh.read()).decode('utf-8')
        except Exception as e:
            print(f'    ERROR loading {ff}: {e}')
    print(f'  Loaded {len(frames_data)}/{len(selected)} frames OK')
    return frames_data

# ================================================================
#  CHECKPOINT
# ================================================================
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r') as f: data = json.load(f)
            print(f'  Checkpoint: {len(data.get("completed_videos", []))} videos done.')
            return data
        except Exception as e:
            print(f'  Could not read checkpoint ({e}) - starting fresh.')
    return {'completed_videos': [], 'results': {}}

def save_checkpoint(data):
    os.makedirs(SAVE_DIR, exist_ok=True)
    tmp = CHECKPOINT_FILE + '.tmp'
    with open(tmp, 'w') as f: json.dump(data, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)

# ================================================================
#  OLLAMA REQUEST HELPER
# ================================================================
def make_ollama_request_robust(messages, temperature=0.0, max_retries=5, base_wait=5):
    import random
    payload = {
        'model': MODEL_NAME,
        'messages': messages,
        'stream': False,
        'options': {
            'temperature': temperature,
            'num_predict': 4096,
            'num_gpu': 99,
            'num_thread': 8,
            'num_batch': 512,
            'num_ctx': 8192,
            'low_vram': False,
            'f16_kv': True,
            'use_mmap': True,
            'use_mlock': False,
        }
    }
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=300)
            if response.status_code == 200:
                data = response.json()
                return data.get('message', {}).get('content', str(data))
            if response.status_code >= 500:
                wait = min(base_wait * (2 ** (attempt-1)) + random.uniform(0,2), 120)
                print(f'[Retry {attempt}/{max_retries}] HTTP {response.status_code}. Waiting {wait:.1f}s ...')
                time.sleep(wait); continue
            return f'ERROR: HTTP {response.status_code}: {response.text[:200]}'
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            wait = min(base_wait * (2 ** (attempt-1)) + random.uniform(0,2), 120)
            print(f'[Retry {attempt}/{max_retries}] Network error: {e}. Waiting {wait:.1f}s ...')
            time.sleep(wait)
        except Exception as e:
            if attempt < max_retries: time.sleep(base_wait * attempt)
            else: return f'ERROR: {e}'
    return f'ERROR: All {max_retries} attempts exhausted'

def frames_to_ollama_content(frames_data, frame_names, prompt_text):
    images = [frames_data[fname] for fname in frame_names if fname in frames_data]
    return {'role': 'user', 'content': prompt_text, 'images': images}

# ================================================================
#  REACT ANALYZER
# ================================================================
class ReActOllamaAnalyzer:
    """
    ReAct (Reasoning + Acting) crime analysis using gemma4:31b via Ollama.
    Thought 1 is text-only; Action 1 processes frames in batches (images);
    remaining Thought/Action/Final steps are text-only multi-turn.
    """
    def _text_call(self, messages, temperature=0.0):
        return make_ollama_request_robust(messages, temperature=temperature)

    def analyze_frames(self, frames_data, video_id, crime_type):
        print(f'\n  [ReAct] {video_id} | {len(frames_data)} frames ...')
        frame_names = sorted(frames_data.keys(), key=lambda x: extract_frame_number(x))
        batches = [frame_names[i:i+BATCH_SIZE] for i in range(0, len(frame_names), BATCH_SIZE)]
        react_log = {}
        conversation = []

        # THOUGHT 1: Pre-analysis reasoning (text-only)
        print('    Thought 1: Initial reasoning ...')
        conversation.append({'role': 'user', 'content': (
            f'I need to analyze {len(frames_data)} security camera frames using the ReAct framework.\n\n'
            'THOUGHT 1: Before examining frames, reason about what indicators to look for '
            'to detect criminal activity in security footage. What are key behaviours and visual cues?'
        )})
        thought1 = self._text_call(conversation)
        conversation.append({'role': 'assistant', 'content': thought1})
        react_log['thought1'] = thought1
        print(f'      Response: {len(thought1)} chars')

        # ACTION 1: Batched frame examination (with images)
        print(f'    Action 1 (images): {len(batches)} batches ...')
        batch_obs = []
        for idx, batch in enumerate(batches, 1):
            prompt = (
                f'THOUGHT 1:\n{thought1}\n\n'
                f'ACTION 1 - Examine Frames (Batch {idx}/{len(batches)}): '
                'Based on the reasoning above, what do you observe? '
                'Report people, actions, objects, events.'
            )
            msg = frames_to_ollama_content(frames_data, batch, prompt)
            obs = make_ollama_request_robust([msg], temperature=0.0)
            batch_obs.append(obs)
            print(f'      Batch {idx}: {len(obs)} chars')

        # Consolidate observations
        formatted = '\n\n'.join(f'--- Batch {i+1} ---\n{s}' for i, s in enumerate(batch_obs))
        observation = self._text_call([{'role': 'user', 'content': (
            f'Consolidate these observations from {len(frames_data)} frames:\n{formatted}\n\n'
            'Write a single unified OBSERVATION.'
        )}])
        react_log['action1_batch_obs'] = batch_obs
        react_log['observation1'] = observation
        print(f'      Consolidated observation: {len(observation)} chars')

        conversation.append({'role': 'user', 'content':
            f'ACTION 1 COMPLETE - OBSERVATION:\n{observation}\n\n'
            f'All {len(frames_data)} frames across {len(batches)} batches examined.'})
        conversation.append({'role': 'assistant', 'content':
            'Observation noted. I will now reason about what these findings mean.'})

        # THOUGHT 2, ACTION 2, THOUGHT 3, FINAL ANSWER — text-only
        for label, prompt in [
            ('thought2',
             'THOUGHT 2: What do the observations collectively indicate? '
             'What patterns emerge? Any contradictions or ambiguities?'),
            ('action2',
             'ACTION 2: Focused re-analysis — zero in on the most diagnostic evidence. '
             'What is most critical for determining crime type?'),
            ('thought3',
             'THOUGHT 3: Reason through crime classification. Consider each: Abuse, Arrest, Arson, '
             'Assault, Burglary, Explosion, Fighting, RoadAccidents, Robbery, Shooting, Shoplifting, '
             'Stealing, Vandalism, Normal. Which best fits?'),
            ('final_answer',
             'FINAL ANSWER:\n'
             'PRIMARY CLASSIFICATION: [crime type]\n'
             'CONFIDENCE LEVEL: [0-100%]\n'
             'SEVERITY: [Low/Medium/High/Critical]\n'
             'KEY EVIDENCE:\n- [point 1]\n- [point 2]\n'
             'REASONING SUMMARY: [how ReAct cycles led to conclusion]\n'
             'ALTERNATIVE INTERPRETATIONS: [other explanations]\n'
             'RECOMMENDED LAW ENFORCEMENT RESPONSE: [actions]'),
        ]:
            print(f'    {label} ...')
            conversation.append({'role': 'user', 'content': prompt})
            response = self._text_call(conversation)
            conversation.append({'role': 'assistant', 'content': response})
            react_log[label] = response
            print(f'      Response: {len(response)} chars')

        return {
            'video_id': video_id, 'crime_type': crime_type,
            'frames_analyzed': len(frames_data), 'total_batches': len(batches),
            'batch_size': BATCH_SIZE, 'prompting_technique': 'REACT',
            'model': MODEL_NAME, 'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'react_log': react_log
        }

# ================================================================
#  PARALLEL VIDEO PROCESSING
# ================================================================
def process_all_crime_folders():
    analyzer   = ReActOllamaAnalyzer()
    all_videos = discover_all_videos_and_frames()
    if not all_videos:
        print('No videos found! Verify FRAMES_DIR path.'); return {}
    cp          = load_checkpoint()
    all_results = cp.get('results', {})
    done_set    = set(cp.get('completed_videos', []))
    skipped     = []
    total       = len(all_videos)
    remaining   = {k: v for k, v in all_videos.items() if k not in done_set}
    print(f'\nVideos: total={total} | done={len(done_set)} | remaining={len(remaining)}')

    checkpoint_lock = threading.Lock()
    print_lock      = threading.Lock()

    def _process_one_video(vkey, vinfo):
        with print_lock: print(f'\n  [START] {vkey}')
        try:
            frames = load_frames_for_video(vinfo, FRAME_INTERVAL)
            if not frames: return vkey, None, 'no frames'
            res = analyzer.analyze_frames(frames, vinfo['video_id'], vinfo['crime_type'])
            with checkpoint_lock:
                all_results[vkey] = res
                done_set.add(vkey)
                save_checkpoint({'completed_videos': list(done_set), 'results': all_results})
            with print_lock: print(f'  [DONE]  {vkey}  ({len(done_set)}/{total})')
            return vkey, res, None
        except Exception as e:
            with print_lock: print(f'  [ERROR] {vkey}: {e}')
            with checkpoint_lock:
                save_checkpoint({'completed_videos': list(done_set), 'results': all_results})
            return vkey, None, f'error: {e}'

    print(f'\n  Launching ThreadPoolExecutor with {MAX_WORKERS} parallel workers ...')
    print('  NOTE: Ollama serves requests sequentially; threads queue automatically.')
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_process_one_video, k, v) for k, v in remaining.items()]
        for fut in as_completed(futures):
            vkey, _, err = fut.result()
            if err: skipped.append(f'{vkey} ({err})')

    ts = time.strftime('%Y%m%d_%H%M%S')
    summary = os.path.join(SAVE_DIR, f'react_ollama_summary_{ts}.json')
    with open(summary, 'w') as f: json.dump(all_results, f, indent=2)
    if skipped:
        with open(os.path.join(SAVE_DIR, f'skipped_{ts}.txt'), 'w') as f:
            f.write('\n'.join(skipped))
    print(f'\nDone. Summary -> {summary}')
    print(f'  Processed: {len(all_results)} | Skipped: {len(skipped)}')
    return all_results

def run():
    print('ReAct Prompting Crime Video Analysis — Ollama gemma4:31b')
    print('=' * 55)
    if not os.path.exists(FRAMES_DIR):
        print(f'ERROR: Frames directory not found: {FRAMES_DIR}'); return
    results = process_all_crime_folders()
    print('\n' + '=' * 55)
    print('REACT PROMPTING (OLLAMA) COMPLETE!')
    print(f'Videos processed: {len(results)}')
    print('=' * 55)

if __name__ == '__main__':
    run()


ReAct Prompting Crime Video Analysis — Ollama gemma4:31b

=== DISCOVERING FRAMES ===
    Root : C:\Opeyemi\PROMPTS\FRAMES
  Categories : ['Abuse', 'Assault', 'Burglary', 'Explosion', 'Fighting', 'RoadAccidents', 'Robbery', 'Shooting', 'Shoplifting', 'Stealing', 'Vandalism']
    Abuse               : 50 videos
    Assault             : 12 videos
    Burglary            : 100 videos
    Explosion           : 50 videos
    Fighting            : 50 videos
    RoadAccidents       : 150 videos
    Robbery             : 150 videos
    Shooting            : 50 videos
    Shoplifting         : 50 videos
    Stealing            : 100 videos
    Vandalism           : 50 videos
  Total videos ready: 812

Videos: total=812 | done=0 | remaining=812

  Launching ThreadPoolExecutor with 4 parallel workers ...
  NOTE: Ollama serves requests sequentially; threads queue automatically.

  [START] Abuse_Abuse001_x264
  Loading 91 frames (ALL) for Abuse001_x264 ...

  [START] Abuse_Abuse002_x264
  Loading 2

In [ ]:
run()
